# The Works by Shakespeare

Let's see what we can do with Shakespeare's collected body of work.

We will do the whole analysis in **Python**. Every step of a text analysis comes down to a
handful of simple operations, and Python has a short, readable way to express each one:

| What we want to do | How we say it in Python |
|--------------------|-------------------------|
| read a whole file | `open(filename).read()` |
| split the text into lines | `text.splitlines()` |
| split the text into words | `text.split()` |
| keep only the lines that mention a word | `[line for line in lines if word in line]` |
| throw away the lines that mention a word | `[line for line in lines if word not in line]` |
| make everything lower case | `text.lower()` |
| delete punctuation | `str.translate` |
| count how many | `len(...)` |
| count how often each word occurs | a dictionary, `counts[word] = counts[word] + 1` |
| show the most frequent ones first | `sorted(..., reverse=True)` |
| look at the first few items only | slicing, e.g. `items[:20]` |

We can download everything from http://www.gutenberg.org/ebooks/100 in plain text format

In [1]:
%%sh
mkdir -p data
cd data
wget https://www.gutenberg.org/cache/epub/100/pg100.txt 2> /dev/null
cd ..
ls -l data

total 11016
-rw-r--r--@ 1 pmolnar  GSUAD\Domain Users  5638480 Sep  1 03:55 pg100.txt


Before we analyze anything, let's read the file and take a look at it.

There is a lot of "junk":
- There is a lot of other text like legal notices included.
- Special characters and even upper and lower case words will affect the analysis.

In [12]:
from pathlib import Path

FILENAME = Path("data/pg100.txt")

text = FILENAME.read_text(encoding="utf-8", errors="replace")
lines = text.splitlines()

print(f"characters: {len(text):,}")
print(f"lines:      {len(lines):,}")

characters: 5,378,701
lines:      196,398


In [13]:
# print the first 20 lines, so we can see what we are dealing with
for line in lines[:20]:
    print(line)

The Project Gutenberg eBook of The Complete Works of William Shakespeare
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.

Title: The Complete Works of William Shakespeare

Author: William Shakespeare


        
Release date: January 1, 1994 [eBook #100]
                Most recently updated: August 24, 2025

Language: English


### Tricks
- `text.lower()` turns everything into lower case, so that *Love*, *love* and *LOVE* are
  counted as the same word
- `str.translate` strips out punctuation in a single pass, so that *hate* and *hate,*
  are no longer two different words
- `text.split()` cuts the text into words. It splits on *any* run of blank space &mdash;
  spaces, tabs, line breaks &mdash; and quietly drops the empty pieces, so we do not have to
  deal with blank lines separately
- To get rid of entire repeating passages, such as the legal notice that Project Gutenberg
  repeats throughout the file:
    1. copy those lines into a file of their own (`data/legalnotice.txt`)
    2. skip every line of the book that contains one of them
- a **dictionary** does the counting for us. We use the word itself as the key and keep a
  running total as the value: the first time we meet a word we store a 1, every time after
  that we add 1 to what is already there
- `sorted()` puts the result in order. With `key=` we tell it to sort by the count rather
  than by the word, and with `reverse=True` we get the most frequent word first

### Some questions?

- How often do the terms "love", "hate", "murder", "faith" appear in the text?
- What are the most frequent words

Let's start with **"hate"**. First we look for the *lines* that mention it: we keep every
line that contains the word, print the first 20 of them, and then count how many there are
in total.

In [14]:
matching = [line for line in lines if "hate" in line]

for line in matching[:20]:
    print(line)

print()
print("The answer is:", len(matching))

For thou art so possessed with murd’rous hate,
Shall hate be fairer lodged than gentle love?
And do whate’er thou wilt swift-footed Time
Such civil war is in my love and hate,
To bear greater wrong, than hate’s known injury.
  For I must ne’er love him whom thou dost hate.
Then hate me when thou wilt, if ever, now,
Whate’er thy thoughts, or thy heart’s workings be,
But shoot not at me in your wakened hate:
As subject to time’s love or to time’s hate,
Past reason hated as a swallowed bait,
Love is my sin, and thy dear virtue hate,
Breathed forth the sound that said ‘I hate’,
‘I hate’ she altered with an end,
  ‘I hate’, from hate away she threw,
Who hateth thee that I do call my friend,
  But love hate on for now I know thy mind,
The more I hear and see just cause of hate?
In vowing new hate after new love bearing:
Let not your hate encounter with my love,

The answer is: 321


Counting lines is a bit crude &mdash; a line that says "hate" twice still counts once, and a
line may contain *hateful* or *hated* as well. So let's count **words** instead: cut the text
into words, keep the ones that contain "hate", and report how often each of them occurs.

In [15]:
words = text.split()                          # cut the text into words
matching = [w for w in words if "hate" in w]  # keep the ones that contain "hate"

# count how often each of them occurs
counts = {}
for word in matching:
    if word in counts:
        counts[word] = counts[word] + 1   # seen before: add one to the running total
    else:
        counts[word] = 1                  # first time we see this word

# sort by the count, largest first, and show the top 20
ranked = sorted(counts.items(), key=lambda item: item[1], reverse=True)
for word, count in ranked[:20]:
    print(f"{count:7d} {word}")

print()
print("The answer is:", len(matching))

    123 hate
     42 hateful
     29 hate,
     20 hated
     18 Whate’er
     16 hates
     13 hate.
     11 whate’er
      8 hated,
      7 hate;
      6 Whatever
      4 hateth
      4 whatever
      3 hate!
      2 hate’s
      2 hate’,
      2 hate?
      2 hates,
      1 hate:
      1 hate’

The answer is: 327


Not bad. But this could be improved...

What are the most frequent words?

Now we clean up the text properly before counting. The recipe is:

1. drop every line that belongs to the repeated legal notice
2. delete the punctuation characters `. , : ? "`
3. turn everything into lower case
4. cut what is left into words
5. count how often each word occurs
6. print the 30 most frequent ones

### Remove Legal Notice
The legal notice in the text appears multiple times. This could distort the word count, since Shakespear did not use terms like "electronic" or "copyright".

We cut-and-paste the legal notice of the document, and save in a new file.

In [6]:
%%writefile data/legalnotice.txt
<<THIS ELECTRONIC VERSION OF THE COMPLETE WORKS OF WILLIAM
SHAKESPEARE IS COPYRIGHT 1990-1993 BY WORLD LIBRARY, INC., AND IS
PROVIDED BY PROJECT GUTENBERG ETEXT OF ILLINOIS BENEDICTINE COLLEGE
WITH PERMISSION.  ELECTRONIC AND MACHINE READABLE COPIES MAY BE
DISTRIBUTED SO LONG AS SUCH COPIES (1) ARE FOR YOUR OR OTHERS
PERSONAL USE ONLY, AND (2) ARE NOT DISTRIBUTED OR USED
COMMERCIALLY.  PROHIBITED COMMERCIAL DISTRIBUTION INCLUDES BY ANY
SERVICE THAT CHARGES FOR DOWNLOAD TIME OR FOR MEMBERSHIP.>>

Writing data/legalnotice.txt


In [7]:
LEGALNOTICE = Path("data/legalnotice.txt")

# read the passages we want to get rid of, one per line
if LEGALNOTICE.exists():
    patterns = [p for p in LEGALNOTICE.read_text(encoding="utf-8").splitlines() if p.strip()]
else:
    print(f"{LEGALNOTICE} not found -- no lines are filtered out")
    patterns = []

# step 1: keep only the lines that contain none of those passages
kept = [line for line in lines if not any(p in line for p in patterns)]

print(f"{len(lines):,} lines in, {len(kept):,} lines kept")

196,398 lines in, 196,398 lines kept


In [16]:
# step 2: a translation table that deletes these punctuation characters
table = str.maketrans("", "", '.,:?"')

counts = {}
for line in kept:
    clean = line.translate(table).lower()   # steps 2 and 3: no punctuation, all lower case
    for word in clean.split():              # step 4: cut the line into words
        if word in counts:                  # step 5: count them
            counts[word] = counts[word] + 1
        else:
            counts[word] = 1

# step 6: sort by the count, largest first, and show the 30 most frequent words
ranked = sorted(counts.items(), key=lambda item: item[1], reverse=True)
for word, count in ranked[:30]:
    print(f"{count:7d} {word}")

  30429 the
  28450 and
  21629 i
  20647 to
  18837 of
  16214 a
  14121 you
  13154 my
  12392 in
  11746 that
   9689 is
   8930 not
   8536 with
   8143 for
   7959 it
   7824 me
   7569 his
   7284 be
   7056 your
   7037 this
   6813 he
   6710 but
   6241 have
   6156 as
   5821 thou
   5262 so
   5213 will
   5203 him
   4652 what
   4440 her


### A detail worth knowing: invisible characters

This file was written on a Windows machine, so every line ends with an invisible *carriage
return* character. Had we cut the text on the space character alone, that invisible character
would have stayed attached to the last word of every line, and `the` and `the`-with-a-carriage-return
would have been counted as two different words &mdash; for almost 200,000 lines.

`text.split()` treats the carriage return as blank space like any other, so it disappears on
its own and we get the right answer. It is a good reminder that a small detail in the cleaning
step can change the result of the entire analysis.

### Filter Stop Words
Stop words are the words in a stop list (or stoplist or negative dictionary) which are filtered out ("stopped") before or after processing of natural language data (i.e. text) because they are deemed to have little semantic value or are otherwise insignificant for the task at hand.

There is no single universal list of stop words used by all natural language processing (NLP) tools, nor any agreed upon rules for identifying stop words, and indeed not all tools even use such a list. Therefore, any group of words can be chosen as the stop words for a given purpose.

The "general trend in information retrieval systems over time has been from standard use of quite large stop lists (200–300 terms) to very small stop lists (7–12 terms) to no stop list whatsoever". [Wikipedia](https://en.wikipedia.org/wiki/Stop_word)

In [11]:
from stop_words import get_stop_words

# Get English stop words using language code
stop_words = get_stop_words('en')

# Or use the full language name
stop_words = get_stop_words('english')

# Use in text processing
example_text = "The quick brown fox jumps over the lazy dog"
words = example_text.lower().split()
filtered_words = [word for word in words if word not in stop_words]
print(filtered_words)  # ['quick', 'brown', 'fox', 'jumps', 'lazy', 'dog']

['quick', 'brown', 'fox', 'jumps', 'lazy', 'dog']


Shakespear: word frequency without stop words.

In [21]:
filter_counts = {item[0]: item[1] for item in counts.items() if item[0] not in stop_words } 
ranked = sorted(filter_counts.items(), key=lambda item: item[1], reverse=True)
for word, count in ranked[:30]:
    print(f"{count:7d} {word}")

   4348 thy
   3217 thee
   2907 lord
   2898 king
   2748 sir
   2402 enter
   2180 love
   2054 hath
   1885 i’ll
   1629 scene
   1489 ’tis
   1149 speak
   1098 duke
   1076 doth
   1075 time
   1000 queen
    991 th’
    976 heart
    975 art
    910 hear
    880 lady
    867 life
    865 death
    858 fair
    854 sweet
    849 hand
    803 father
    788 true
    774 pray
    768 master


# Scripts with a Hash-Bang
Even in the few examples above we have used the same sequence of steps over and over again. We should turn those steps into a new command (or script) of our own, so that we can reuse them without copying code.

How to create a script:
1. Create a text file with the name of your new "command". We often add something like ".py" to indicate which language the script is written in. E.g. `wordfrequency.py`
2. The very first line of the text file must indicate the interpreter that is going to execute the program. In our case `#!/usr/bin/env python3`
3. The file needs permission to be executed, so that we can use the new script just like any other program

Our script takes the **name of a file** as its argument and prints every word in it together with how often it occurred, most frequent first &mdash; exactly the steps we worked out above.

A program has two things to get right besides the actual work:

- **Was it given what it needs?** Whatever the user typed after the name of the program arrives
  in the list `sys.argv`. Its first entry, `sys.argv[0]`, is the name of the program itself, so
  the file name we are after is `sys.argv[1]`. If it is not there, we print a short *usage*
  message reminding the user how to call the program, and stop.
- **Did anything go wrong?** The file may not exist, we may not be allowed to read it, or it may
  turn out to be a directory. Instead of trying to check for every one of these in advance, we
  simply *try* to read the file and deal with the problem if one comes up. That is what
  `try ... except` is for: the code that might fail goes into the `try` block, and each `except`
  block says what to do about one particular kind of failure.

Messages meant for the person running the program, rather than results, are printed to the *error*
output `sys.stderr` so that they stay out of the way of the word counts. `sys.exit(1)` stops the
program and reports that it did not succeed.

In [ ]:
%%writefile wordfrequency.py
#!/usr/bin/env python3
"""Read a text file, print each word and how often it occurs, most frequent first."""
import sys

table = str.maketrans("", "", '.,:?"')

if __name__ == "__main__":

    # were we given a file name?
    if len(sys.argv) != 2:
        print(f"usage: {sys.argv[0]} FILENAME", file=sys.stderr)
        sys.exit(1)

    filename = sys.argv[1]
    counts = {}

    try:
        with open(filename, encoding="utf-8", errors="replace") as textfile:
            for line in textfile:
                for word in line.translate(table).lower().split():
                    if word in counts:
                        counts[word] = counts[word] + 1
                    else:
                        counts[word] = 1
    except FileNotFoundError:
        print(f"{sys.argv[0]}: there is no file named {filename}", file=sys.stderr)
        sys.exit(1)
    except PermissionError:
        print(f"{sys.argv[0]}: not allowed to read {filename}", file=sys.stderr)
        sys.exit(1)
    except OSError as problem:
        # anything else that went wrong while reading, e.g. it is a directory
        print(f"{sys.argv[0]}: could not read {filename}: {problem}", file=sys.stderr)
        sys.exit(1)

    for word, count in sorted(counts.items(), key=lambda item: item[1], reverse=True):
        print(f"{count:7d} {word}")

Now give the file permission to run, then hand it the name of our book.

In [ ]:
%%sh
chmod +x wordfrequency.py

In [ ]:
%%sh
./wordfrequency.py data/pg100.txt > data/wordcounts.txt
head -30 data/wordcounts.txt

Let's check the two safety nets: calling the script without a file name, and asking it for a
file that does not exist. Neither should produce an error message from Python itself.

In [ ]:
%%sh
./wordfrequency.py
echo "exit code: $?"

./wordfrequency.py data/does_not_exist.txt
echo "exit code: $?"

In [10]:
#! pip install stop-words